In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Import liberies**

In [ ]:
#  Basic Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

#  TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#  Keras Pre-trained CNN Models
from tensorflow.keras.applications import (
    ResNet50, ResNet101,
    EfficientNetB3, EfficientNetB0,
    MobileNetV2, DenseNet121,
    InceptionV3, Xception,
    NASNetMobile, VGG16
)

#  Model Utilities
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

#  Misc
import warnings
warnings.filterwarnings('ignore')

# **Load & Preprosess**

In [ ]:
base_dir = "/kaggle/input/skin-cancer-mnist-ham10000"
csv_path = os.path.join(base_dir, "HAM10000_metadata.csv")
img_dir = os.path.join(base_dir, "HAM10000_images_part_1")
img_dir = os.path.join(base_dir, "HAM10000_images_part_2")

df = pd.read_csv(csv_path)
df["image_path"] = df["image_id"].apply(lambda x: os.path.join(img_dir, f"{x}.jpg"))
df = df[df["image_path"].apply(os.path.exists)]

# Use 'dx' (diagnosis) as label
df["label"] = df["dx"].astype(str)  # Ensure labels are strings
labels = df["label"].unique()
label_map = {label: idx for idx, label in enumerate(labels)}
df["label_idx"] = df["label"].map(label_map)

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

In [ ]:
# Define dataset path
dataset_path = "/kaggle/input/skin-cancer-mnist-ham10000"
metadata = pd.read_csv(os.path.join(dataset_path, "HAM10000_metadata.csv"))
num_examples = len(metadata)
print(f"✅ Dataset loaded: {num_examples} samples")
metadata.head()

# **Get label from metadata**

In [ ]:
import os
from glob import glob
import random
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

# -----------------------------
# Load metadata
# -----------------------------
metadata_path = '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
metadata = pd.read_csv(metadata_path)

# -----------------------------
# Create image path dictionary
# -----------------------------
dataset_path = '/kaggle/input/skin-cancer-mnist-ham10000'
image_path = {}
for part in ['HAM10000_images_part_1', 'HAM10000_images_part_2']:
    folder_path = os.path.join(dataset_path, part)
    part_images = {os.path.splitext(os.path.basename(x))[0]: x
                   for x in glob(os.path.join(folder_path, '*.jpg'))}
    image_path.update(part_images)

print(f"Found {len(image_path)} images")

# -----------------------------
# Select up to 9 random images
# -----------------------------
num_samples = min(9, len(image_path))
random_ids = random.sample(list(image_path.keys()), num_samples)

# -----------------------------
# Plot images with labels
# -----------------------------
plt.figure(figsize=(8, 8))
for i, img_id in enumerate(random_ids):
    img_path = image_path[img_id]
    img = Image.open(img_path)
    
    # Get label from metadata
    row = metadata[metadata['image_id'] == img_id]
    label = row['dx'].iloc[0] if not row.empty else 'Unknown'
    
    plt.subplot(3, 3, i+1)
    plt.imshow(img)
    plt.title(f'{label}\nID: {img_id}')
    plt.axis('off')

plt.tight_layout()
plt.show()


# **EDA (Basic Info & Visualizations**

# Basic Info & Null

In [ ]:
print("Dataset Shape:", df.shape)
print("Null Values:\n", df.isnull().sum())

# Lesion Type Distribution

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=df, x='dx', order=df['dx'].value_counts().index)
plt.title("Lesion Type Distribution")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

# Sex Distribution

In [ ]:
sns.countplot(data=df, x='sex')
plt.title("Sex Distribution")
plt.grid(True)
plt.show()

# Age Distribution

In [ ]:
sns.histplot(df['age'].dropna(), bins=30, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.grid(True)
plt.show()

# **Anatomical Site Distribution**

In [ ]:
sns.countplot(data=df, x='localization', order=df['localization'].value_counts().index)
plt.title("Anatomical Location Distribution")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

# **Balance**

In [ ]:
label_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
metadata['label'] = metadata['dx'].map(lambda x: label_names.index(x))
print("Imbalanced dataset distribution:")
print(metadata['label'].value_counts())

In [ ]:
# Add image paths
image_path = {os.path.splitext(os.path.basename(x))[0]: x
             for x in glob(os.path.join(dataset_path, '*', '*.jpg'))}
metadata['path'] = metadata['image_id'].map(image_path.get)

# **Split data**

In [ ]:
# SPLIT FIRST - by unique images to avoid any overlap
train_df, test_df = train_test_split(
    metadata, 
    test_size=0.2, 
    random_state=42, 
    stratify=metadata['label']
)
train_df, val_df = train_test_split(
    train_df, 
    test_size=0.25, 
    random_state=42, 
    stratify=train_df['label']
)

print(f"\nAfter splitting:")
print(f"Train: {len(train_df)} samples")
print(f"Val: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

In [ ]:
# NOW BALANCE ONLY THE TRAINING SET
data_aug_rate = [15, 10, 5, 50, 5, 1, 40]

def balance_training_data(train_df, data_aug_rate):
    """Balance only the training data by class"""
    balanced_dfs = []
    
    for class_idx, rate in enumerate(data_aug_rate):
        class_df = train_df[train_df['label'] == class_idx]
        if len(class_df) > 0 and rate > 0:
            # Multiply the class by the rate
            augmented_df = pd.concat([class_df] * rate, ignore_index=True)
            balanced_dfs.append(augmented_df)
        elif len(class_df) > 0:
            # Rate 0 means use original
            balanced_dfs.append(class_df)
    
    return pd.concat(balanced_dfs, ignore_index=True)

# Balance only training data
train_df_balanced = balance_training_data(train_df, data_aug_rate)

print(f"\nBalanced training set distribution:")
print(train_df_balanced['label'].value_counts().sort_index())
print(f"Balanced training set size: {len(train_df_balanced)}")

# Validation and test sets remain unbalanced (original distribution)
print(f"\nValidation set distribution:")
print(val_df['label'].value_counts().sort_index())
print(f"\nTest set distribution:")
print(test_df['label'].value_counts().sort_index())# Verify no image overlap between sets
def check_image_overlap(train_df, val_df, test_df):
    """Check if there are any overlapping images between train/val/test"""
    train_imgs = set(train_df['image_id'].unique())
    val_imgs = set(val_df['image_id'].unique())
    test_imgs = set(test_df['image_id'].unique())
    
    train_val_overlap = train_imgs.intersection(val_imgs)
    train_test_overlap = train_imgs.intersection(test_imgs)
    val_test_overlap = val_imgs.intersection(test_imgs)
    
    print(f"\nImage overlap check:")
    print(f"Train-Val overlap: {len(train_val_overlap)} images")
    print(f"Train-Test overlap: {len(train_test_overlap)} images")
    print(f"Val-Test overlap: {len(val_test_overlap)} images")
    
    if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
        print("✓ No image overlap between sets - clean split!")
    else:
        print("⚠ Warning: Image overlap detected!")

# Run the overlap check
check_image_overlap(train_df_balanced, val_df, test_df)

# **Data Generator**

In [ ]:
import os
from glob import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# -----------------------------
# Paths
# -----------------------------
dataset_path = '/kaggle/input/skin-cancer-mnist-ham10000'
metadata_path = os.path.join(dataset_path, 'HAM10000_metadata.csv')

# -----------------------------
# Load metadata
# -----------------------------
metadata = pd.read_csv(metadata_path)

# -----------------------------
# Create full image paths
# -----------------------------
image_path = {}
for part in ['HAM10000_images_part_1', 'HAM10000_images_part_2']:
    folder_path = os.path.join(dataset_path, part)
    part_images = {os.path.splitext(os.path.basename(x))[0]: x
                   for x in glob(os.path.join(folder_path, '*.jpg'))}
    image_path.update(part_images)

# Add image_path column to metadata
metadata['image_path'] = metadata['image_id'].map(image_path)
metadata = metadata.dropna(subset=['image_path'])  # drop rows without images

# Make sure label column is string
metadata['label'] = metadata['dx'].astype(str)

# -----------------------------
# Train/Validation Split
# -----------------------------
train_df, val_df = train_test_split(metadata, test_size=0.2, stratify=metadata['label'], random_state=42)

# -----------------------------
# ImageDataGenerators
# -----------------------------
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=20,
                                   width_shift_range=0.1,
                                   height_shift_range=0.1,
                                   horizontal_flip=True,
                                   vertical_flip=True)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path',
    y_col='label',
    target_size=IMAGE_SIZE,
    class_mode='sparse',
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_gen = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_path',
    y_col='label',
    target_size=IMAGE_SIZE,
    class_mode='sparse',
    batch_size=BATCH_SIZE,
    shuffle=False
)


# **Model Builder**

In [ ]:
num_classes = df['label'].nunique()

def build_model(base_model_class, img_size=(224, 224), num_classes=7, dropout_rate=0.3):
    if isinstance(img_size, int):
        img_size = (img_size, img_size)
        
    base_model = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=(*img_size, 3)
    )
    base_model.trainable = False  # freeze backbone initially

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model, base_model


# **Callbacks**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

es = EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1)
checkpoint = ModelCheckpoint("best_model.h5", monitor='val_accuracy', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

callbacks_list = [es, checkpoint, reduce_lr]


# **Train Models**

In [ ]:
models_to_train = {
    "EfficientNetB0": EfficientNetB0,
    "EfficientNetB3": EfficientNetB3,
    "ResNet50": ResNet50,
    "ResNet101": ResNet101,
    "DenseNet121": DenseNet121,
    "MobileNetV2": MobileNetV2,
    "InceptionV3": InceptionV3,
    "Xception": Xception
}

history_dict = {}

for name, model_class in models_to_train.items():
    print(f"\n🔹 Training {name}...")
    model, base_model = build_model(model_class, img_size=IMAGE_SIZE, num_classes=num_classes)

    # Initial training with frozen backbone
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=25,
        callbacks=[es, checkpoint, reduce_lr],
        verbose=1
    )

    # Fine-tuning top layers
    base_model.trainable = True
    for layer in base_model.layers[:-50]:  # freeze all except top 50 layers
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n🔹 Fine-tuning {name}...")
    fine_tune_history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=7,
        callbacks=[es, checkpoint, reduce_lr],
        verbose=1
    )

    history_dict[name] = history
    model.save(f"{name}_HAM10000.h5")
    print(f" Saved {name} as {name}_HAM10000.h5")


# **Save models**

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/HAM10000_models", 'zip', "/kaggle/working")

In [ ]:
import tensorflow as tf

# Train your model
model = tf.keras.Sequential([...])  # Define your model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10)

# Save the model
model.save('/kaggle/working/my_model.h5')


In [ ]:
from IPython.display import FileLink

FileLink('/kaggle/working/my_model.pkl')  # Replace with your filename


# **Prepare test generator**

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_directory(
    '/kaggle/input/ham10000-test-folder',  
    target_size=(224, 224),                
    batch_size=32,
    class_mode='sparse',
    shuffle=False
)


# **Predictions & Ground Truth**

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model

# Ground truth labels from test generator
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

# Dictionary to store predictions for each model
predictions_dict = {}

# Loop through all saved models
MODEL_PATHS = {
    "EfficientNetB0": "/kaggle/working/EfficientNetB0_HAM10000.h5",
    "EfficientNetB3": "/kaggle/working/EfficientNetB3_HAM10000.h5", 
    "ResNet50": "/kaggle/working/ResNet50_HAM10000.h5",
    "ResNet101": "/kaggle/working/ResNet101_HAM10000.h5",
    "DenseNet121": "/kaggle/working/DenseNet121_HAM10000.h5",
    "MobileNetV2": "/kaggle/working/MobileNetV2_HAM10000.h5",
    "InceptionV3": "/kaggle/working/InceptionV3_HAM10000.h5",
    "Xception": "/kaggle/working/Xception_HAM10000.h5"
}

for name, path in MODEL_PATHS.items():
    print(f"\n🔹 Predicting with {name}...")
    model = load_model(path)
    
    # Predict probabilities on test set
    y_pred_prob = model.predict(test_gen, verbose=1)
    
    # Convert probabilities to class indices
    y_pred = np.argmax(y_pred_prob, axis=1)
    
    # Store predictions
    predictions_dict[name] = y_pred

print("\n Predictions completed for all models.")


# **Comparison & Evaluation**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Dictionary to store overall test accuracy
accuracy_dict = {}

for name, y_pred in predictions_dict.items():
    print(f"\n📊 Evaluation for {name}:")

    # Overall test accuracy
    acc = accuracy_score(y_true, y_pred)
    accuracy_dict[name] = acc
    print(f"✅ Test Accuracy: {acc*100:.2f}%\n")

    # Classification report
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, 
                yticklabels=class_names, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

# Optional: Compare all model accuracies in a bar plot
plt.figure(figsize=(10,6))
plt.bar(accuracy_dict.keys(), [v*100 for v in accuracy_dict.values()], color='skyblue')
plt.ylabel('Test Accuracy (%)')
plt.title('Test Accuracy Comparison Across Models')
plt.xticks(rotation=45)
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.show()

# **Real-world test**

In [ ]:
# =====================================================
# 1. Imports & Setup
# =====================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model

import wandb
from wandb.keras import WandbCallback

# =====================================================
# 2. Configurations
# =====================================================
IMG_SIZE = (224, 224)
IMG_FOLDER = "/kaggle/input/samplefortest"  # test folder
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_DESCRIPTIONS = {
    'akiec': 'Actinic keratoses and intraepithelial carcinoma',
    'bcc': 'Basal cell carcinoma', 
    'bkl': 'Benign keratosis-like lesions',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic nevi',
    'vasc': 'Vascular lesions'
}

# Your dictionary of models to train (already defined in your loop)
models_to_train = {
    "EfficientNetB0": tf.keras.applications.EfficientNetB0,
    "EfficientNetB3": tf.keras.applications.EfficientNetB3,
    "ResNet50": tf.keras.applications.ResNet50,
    "ResNet101": tf.keras.applications.ResNet101,
    "DenseNet121": tf.keras.applications.DenseNet121,
    "MobileNetV2": tf.keras.applications.MobileNetV2,
    "InceptionV3": tf.keras.applications.InceptionV3,
    "Xception": tf.keras.applications.Xception
}

# =====================================================
# 3. Build model function
# =====================================================
def build_model(model_class, img_size, num_classes):
    base_model = model_class(
        weights="imagenet", 
        include_top=False, 
        input_shape=img_size + (3,)
    )
    base_model.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
    output = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, base_model

# =====================================================
# 4. Start W&B project
# =====================================================
wandb.init(
    project="HAM10000_multimodel",
    config={
        "epochs_frozen": 25,
        "epochs_finetune": 7,
        "batch_size": train_gen.batch_size,
        "image_size": IMG_SIZE,
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "dataset": "HAM10000"
    }
)

# Callbacks
es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, verbose=1)
os.makedirs("checkpoints", exist_ok=True)

# =====================================================
# 5. Training loop
# =====================================================
history_dict = {}
MODEL_PATHS = {}

for name, model_class in models_to_train.items():
    print(f"\n🔹 Training {name}...")
    model, base_model = build_model(model_class, img_size=IMG_SIZE, num_classes=num_classes)

    checkpoint = ModelCheckpoint(
        filepath=f"checkpoints/{name}_best.h5",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )

    # Step 1: Train frozen backbone
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=wandb.config.epochs_frozen,
        callbacks=[es, checkpoint, reduce_lr, WandbCallback()],
        verbose=1
    )

    # Step 2: Fine-tune top 50 layers
    base_model.trainable = True
    for layer in base_model.layers[:-50]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print(f"\n🔹 Fine-tuning {name}...")
    fine_tune_history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=wandb.config.epochs_finetune,
        callbacks=[es, checkpoint, reduce_lr, WandbCallback()],
        verbose=1
    )

    # Save final model
    model_path = f"/kaggle/working/{name}_HAM10000.h5"
    model.save(model_path)
    MODEL_PATHS[name] = model_path
    print(f"✅ Saved {name} as {model_path}")

    # Save histories
    history_dict[name] = {
        "frozen": history.history,
        "finetune": fine_tune_history.history
    }

wandb.finish()

# =====================================================
# 6. Load models for testing
# =====================================================
models = {}
for name, path in MODEL_PATHS.items():
    try:
        models[name] = load_model(path)
        print(f"✅ Loaded {name}")
    except Exception as e:
        print(f"❌ Failed to load {name}: {e}")

# =====================================================
# 7. Preprocess & Predict
# =====================================================
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    x = image.img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)
    return x, img

image_files = [f for f in os.listdir(IMG_FOLDER) if f.lower().endswith(('.jpg','.png','.jpeg'))]

for img_file in image_files:
    img_path = os.path.join(IMG_FOLDER, img_file)
    x, pil_img = preprocess_image(img_path)

    all_model_preds = []
    print(f"\nImage: {img_file}")

    for name, model in models.items():
        preds = model.predict(x, verbose=0)[0]
        top_index = np.argmax(preds)
        top_class = CLASS_NAMES[top_index]
        confidence = preds[top_index]
        all_model_preds.append(top_class)
        print(f"{name}: {top_class} ({confidence*100:.2f}%)")

    # Consensus
    consensus = Counter(all_model_preds).most_common(1)[0]
    final_class, votes = consensus

    plt.figure(figsize=(6,6))
    plt.imshow(pil_img)
    plt.axis("off")
    plt.title(f"Consensus Prediction: {final_class} ({votes}/{len(models)} votes)\n{CLASS_DESCRIPTIONS[final_class]}")
    plt.show()
